In [23]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

from sklearn.metrics.pairwise import euclidean_distances


import json

from mistralai import Mistral

from pathlib import Path

# Preliminari

In [2]:
# Configurazione Mistral
client = Mistral(
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent

MODEL = constants.MISTRAL_EMBED
RESULTS_FILE_NAME = 'mistral_embeddings.jsonl'

# Load Data

In [3]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

train_data, validation_data, test_data = data['train'], data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(train_data.id) & set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")

len(train_data) = 184
len(validation_data) = 66
len(test_data) = 65


# Generazione

In [5]:
# Funzione generatrice
def generate_embedding(model: str, text: str | list[str]):
    response = client.embeddings.create(
        model=model,
        inputs=text
    )
    return response

In [ ]:
# Esempio
if True:
    embeddings = generate_embedding(MODEL, ['Avengers... Uniti!', "Thor è l'avenger più forte!", 'La cina è un paese enorme!'])

In [48]:
x = generate_embedding(MODEL, 'Avengers... Uniti!')

In [52]:
x.data[0].embedding

[0.010223388671875,
 0.0186309814453125,
 0.052642822265625,
 0.007678985595703125,
 0.0261993408203125,
 0.042236328125,
 0.0008611679077148438,
 -0.0066375732421875,
 -0.0035572052001953125,
 -0.04315185546875,
 0.02044677734375,
 0.056488037109375,
 -0.025421142578125,
 -0.00745391845703125,
 -0.0218048095703125,
 0.0302734375,
 0.00141143798828125,
 0.0006108283996582031,
 0.0295867919921875,
 0.0305023193359375,
 0.010162353515625,
 -0.0123138427734375,
 -0.057159423828125,
 -0.009033203125,
 -0.005168914794921875,
 0.0164947509765625,
 -0.021240234375,
 -0.056915283203125,
 -0.054229736328125,
 -0.0181884765625,
 -0.0014257431030273438,
 -0.039306640625,
 0.0224761962890625,
 -0.006549835205078125,
 0.0379638671875,
 0.01422882080078125,
 -0.01580810546875,
 -0.048126220703125,
 0.05511474609375,
 -0.04290771484375,
 0.00672149658203125,
 -0.03253173828125,
 0.0058441162109375,
 -0.02191162109375,
 0.0012493133544921875,
 -0.036834716796875,
 -0.001178741455078125,
 0.00892639160

In [29]:
if True: # To see the full output
    pprint(embeddings.data)
    print(embeddings.data[0].embedding)

[EmbeddingResponseData(object='embedding', embedding=[0.0102691650390625, 0.01873779296875, 0.052825927734375, 0.007843017578125, 0.0262908935546875, 0.04266357421875, 0.0007228851318359375, -0.006687164306640625, -0.0036678314208984375, -0.0433349609375, 0.02008056640625, 0.056182861328125, -0.0251617431640625, -0.0076751708984375, -0.0215606689453125, 0.030242919921875, 0.0015163421630859375, 0.0007371902465820312, 0.0293426513671875, 0.03070068359375, 0.01021575927734375, -0.01241302490234375, -0.056884765625, -0.0091400146484375, -0.004486083984375, 0.016357421875, -0.0207672119140625, -0.056640625, -0.05438232421875, -0.0179443359375, -0.0017070770263671875, -0.039031982421875, 0.0224609375, -0.006572723388671875, 0.03814697265625, 0.01393890380859375, -0.0157928466796875, -0.0478515625, 0.054840087890625, -0.042877197265625, 0.006855010986328125, -0.03228759765625, 0.006092071533203125, -0.0217742919921875, 0.0009026527404785156, -0.03656005859375, -0.0015726089477539062, 0.00891

In [30]:
print(euclidean_distances([embeddings.data[0].embedding], [embeddings.data[1].embedding]))
print(euclidean_distances([embeddings.data[0].embedding], [embeddings.data[2].embedding]))
print(euclidean_distances([embeddings.data[1].embedding], [embeddings.data[2].embedding]))

[[0.63826523]]
[[0.82261679]]
[[0.73582593]]


## EMBEDDINGS

In [53]:
print(MODEL)
df = pd.concat([train_data, validation_data, test_data], ignore_index=True)
ids = []
splits = []
embeddings = []
for i, row in df.iterrows():
    row = df.iloc[i]
    embedding = generate_embedding(MODEL, row[constants.REPORT_COLUMN_NAME])
    embeddings.append(embedding.data[0].embedding)
    splits.append(row['split'])
    ids.append(int(row[constants.ID_COMULM_NAME]))

mistral-embed-2312


In [ ]:
assert len(ids) == len(splits) == len(embeddings)

In [58]:
results_dicts = []
assert len(ids) == len(splits) == len(embeddings)
for id, split, embedding, in zip(ids, splits, embeddings):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': id,
            'embedding': embedding
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'embeddings' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')